<a href="https://colab.research.google.com/github/rodrigologin0-cpu/Rodrigo-de-Souza-Lima/blob/main/Calculo_Carepa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CONFIGURAÇÃO DO FORNO
# ============================================================

ZONAS = [
    {"zona": "Pre-aquecimento", "comprimento_m": 7.59},
    {"zona": "Aquecimento_1",   "comprimento_m": 9.51},
    {"zona": "Aquecimento_2",   "comprimento_m": 12.17},
    {"zona": "Encharque",       "comprimento_m": 5.85},
]

TEMP_LIMIAR_C = 800.0

# Parâmetros do modelo relativo
# beta controla o quanto a temperatura pesa.
# Quanto maior beta, mais agressivo fica o efeito da temperatura.
BETA_TEMP = 0.006

# Referências para normalização
TEMP_REF_C = 1100.0
O2_REF_PCT = 2.0

# Calibração: condição normal do forno gera aprox. 1,2 mm
CAREPA_REF_MM = 1.2


In [ ]:
# ============================================================
# MODELO DE TAXA RELATIVA DE CAREPA
# ============================================================

def fator_temperatura(temp_c):
    """
    Fator térmico.
    Abaixo de 800 °C, considera formação desprezível.
    Acima de 800 °C, crescimento exponencial relativo.
    """
    temp_c = np.asarray(temp_c)

    fator = np.zeros_like(temp_c, dtype=float)
    ativo = temp_c >= TEMP_LIMIAR_C

    fator[ativo] = np.exp(BETA_TEMP * (temp_c[ativo] - TEMP_REF_C))

    return fator


def fator_o2(o2_pct):
    """
    Fator de oxigênio.
    Usa raiz quadrada porque o documento cita dependência pela raiz do teor de O2.
    """
    o2_pct = np.asarray(o2_pct)
    o2_seguro = np.maximum(o2_pct, 0.01)

    return np.sqrt(o2_seguro / O2_REF_PCT)


def taxa_carepa_relativa(temp_c, o2_pct):
    """
    Taxa relativa instantânea de formação de carepa.
    Unidade: índice relativo por minuto.
    """
    return fator_temperatura(temp_c) * fator_o2(o2_pct)



In [ ]:
# ============================================================
# CÁLCULO POR PLACA
# ============================================================

def calcular_carepa_por_placa(df):
    """
    Espera um DataFrame com colunas:

    slab_id              -> identificação da placa
    timestamp            -> data/hora
    zona                 -> nome da zona
    temp_superficie_C    -> temperatura estimada da superfície da placa
    o2_pct               -> oxigênio médio da zona/região

    O cálculo integra a taxa ao longo do tempo para cada placa.
    """

    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["slab_id", "timestamp"])

    resultados = []

    for slab_id, g in df.groupby("slab_id"):
        g = g.sort_values("timestamp").copy()

        # delta de tempo em minutos
        g["dt_min"] = g["timestamp"].diff().dt.total_seconds() / 60.0
        g["dt_min"] = g["dt_min"].fillna(0)

        g["taxa_relativa"] = taxa_carepa_relativa(
            g["temp_superficie_C"].values,
            g["o2_pct"].values
        )

        g["incremento_indice"] = g["taxa_relativa"] * g["dt_min"]

        indice_total = g["incremento_indice"].sum()

        resultados.append({
            "slab_id": slab_id,
            "inicio": g["timestamp"].min(),
            "fim": g["timestamp"].max(),
            "tempo_total_min": (g["timestamp"].max() - g["timestamp"].min()).total_seconds() / 60.0,
            "temp_media_C": g["temp_superficie_C"].mean(),
            "temp_max_C": g["temp_superficie_C"].max(),
            "o2_medio_pct": g["o2_pct"].mean(),
            "indice_carepa_bruto": indice_total
        })

    return pd.DataFrame(resultados)


def calibrar_para_mm(df_resultado, indice_referencia=None):
    """
    Converte o índice bruto em mm estimado.
    Se indice_referencia não for informado, usa a mediana do lote como referência.
    """

    df = df_resultado.copy()

    if indice_referencia is None:
        indice_referencia = df["indice_carepa_bruto"].median()

    df["carepa_estimada_mm"] = CAREPA_REF_MM * (
        df["indice_carepa_bruto"] / indice_referencia
    )

    df["kpi_carepa"] = 100.0 * (
        df["carepa_estimada_mm"] / CAREPA_REF_MM
    )

    return df, indice_referencia



In [ ]:

# ============================================================
# EXEMPLO DE USO COM DADOS SIMULADOS
# ============================================================

def gerar_exemplo():
    dados = []

    placas = ["P001", "P002", "P003"]

    for i, slab in enumerate(placas):
        inicio = pd.Timestamp("2026-04-30 08:00:00") + pd.Timedelta(minutes=10*i)

        tempo = 0

        perfil = [
            ("Pre-aquecimento", 70, 780 + 20*i, 2.8),
            ("Aquecimento_1",   55, 980 + 10*i, 2.5),
            ("Aquecimento_2",   50, 1120 + 15*i, 2.2),
            ("Encharque",       35, 1180 + 10*i, 1.8),
        ]

        for zona, duracao_min, temp_base, o2 in perfil:
            for t in range(0, duracao_min, 5):
                dados.append({
                    "slab_id": slab,
                    "timestamp": inicio + pd.Timedelta(minutes=tempo + t),
                    "zona": zona,
                    "temp_superficie_C": temp_base + np.random.normal(0, 8),
                    "o2_pct": o2 + np.random.normal(0, 0.15)
                })

            tempo += duracao_min

    return pd.DataFrame(dados)



In [ ]:
# ============================================================
# RODAR
# ============================================================

df = gerar_exemplo()

resultado = calcular_carepa_por_placa(df)

resultado_calibrado, indice_ref = calibrar_para_mm(resultado)

print("\nÍndice de referência usado:", indice_ref)
print(resultado_calibrado)



In [ ]:
# ============================================================
# GRÁFICO KPI POR PLACA
# ============================================================

plt.figure(figsize=(10, 5))
plt.bar(resultado_calibrado["slab_id"], resultado_calibrado["kpi_carepa"])
plt.axhline(100, linestyle="--")
plt.xlabel("Placa")
plt.ylabel("KPI Carepa")
plt.title("KPI relativo de formação de carepa por placa")
plt.grid(True, axis="y")
plt.show()


In [ ]:
# ============================================================
# GRÁFICO CAREPA ESTIMADA EM MM
# ============================================================

plt.figure(figsize=(10, 5))
plt.bar(resultado_calibrado["slab_id"], resultado_calibrado["carepa_estimada_mm"])
plt.axhline(1.2, linestyle="--")
plt.xlabel("Placa")
plt.ylabel("Carepa estimada [mm]")
plt.title("Estimativa de espessura de carepa por placa")
plt.grid(True, axis="y")
plt.show()